# 10 — Repeated Group-Aware Validation and Stability

Evaluate the stability of Notebook 03's random-row versus host-star-grouped comparison across multiple random split seeds.

**Reproducibility note:** This notebook follows Notebook 03's primary sample definition and Random Forest pipeline. It does not overwrite or modify any earlier notebook. Run cells from top to bottom in Kaggle or Jupyter.

In [1]:
import os, glob, warnings
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, GroupKFold
from sklearn.metrics import (
    roc_auc_score, average_precision_score, balanced_accuracy_score,
    brier_score_loss, precision_score, recall_score, f1_score, log_loss
)

# Match Notebook 03's primary population and modeling settings.
RADIUS_MIN, RADIUS_MAX = 1.0, 4.0
TEFF_MIN, TEFF_MAX = 4700, 6500
N_SPLITS = 5
THRESHOLD = 0.50
RANDOM_STATE = 42
N_ESTIMATORS = 400

# Set this manually if auto-detection finds zero or multiple candidate CSV files.
DATA_PATH = None  # e.g. "/kaggle/input/your-dataset/cumulative.csv"

def locate_koi_csv():
    if DATA_PATH is not None:
        if not os.path.exists(DATA_PATH):
            raise FileNotFoundError(f"DATA_PATH does not exist: {DATA_PATH}")
        return DATA_PATH
    candidates = sorted(
        p for p in glob.glob("/kaggle/input/**/*.csv", recursive=True)
        if ("cumulative" in os.path.basename(p).lower()
            or "koi" in os.path.basename(p).lower())
    )
    if len(candidates) == 1:
        return candidates[0]
    if not candidates:
        raise FileNotFoundError(
            "No KOI cumulative CSV found under /kaggle/input. "
            "Attach the same catalog used in Notebook 03 and set DATA_PATH manually."
        )
    print("Possible KOI CSV files:")
    for p in candidates:
        print(" -", p)
    raise ValueError("Multiple possible KOI CSVs found. Set DATA_PATH explicitly.")

def load_main_sample():
    path = locate_koi_csv()
    print("Using catalog:", path)
    raw = pd.read_csv(path, comment="#", low_memory=False)
    required = {"kepid", "koi_disposition", "koi_prad", "koi_steff"}
    missing = required - set(raw.columns)
    if missing:
        raise KeyError(f"Required columns missing: {sorted(missing)}")
    for col in ["kepid", "koi_prad", "koi_steff"]:
        raw[col] = pd.to_numeric(raw[col], errors="coerce")
    catalog = raw[
        raw["koi_disposition"].isin(["CONFIRMED", "FALSE POSITIVE", "CANDIDATE"])
        & raw["kepid"].notna()
        & raw["koi_prad"].between(RADIUS_MIN, RADIUS_MAX, inclusive="both")
        & raw["koi_steff"].between(TEFF_MIN, TEFF_MAX, inclusive="both")
    ].copy()
    catalog["group"] = catalog["kepid"].astype("int64")
    labeled = catalog[
        catalog["koi_disposition"].isin(["CONFIRMED", "FALSE POSITIVE"])
    ].copy()
    labeled["label"] = (labeled["koi_disposition"] == "CONFIRMED").astype(int)
    candidate_features = [
        "koi_period", "koi_time0bk", "koi_impact", "koi_duration",
        "koi_depth", "koi_prad", "koi_teq", "koi_insol",
        "koi_steff", "koi_slogg", "koi_srad", "koi_kepmag"
    ]
    features = [c for c in candidate_features if c in labeled.columns]
    if len(features) < 3:
        raise ValueError(f"Fewer than 3 Notebook 03 features available: {features}")
    X = labeled[features].apply(pd.to_numeric, errors="coerce")
    X = X.replace([np.inf, -np.inf], np.nan)
    drop = [c for c in X if X[c].notna().sum() == 0 or X[c].nunique(dropna=True) <= 1]
    if drop:
        print("Dropping all-missing/constant features:", drop)
        X = X.drop(columns=drop)
    y = labeled["label"].to_numpy(dtype=int)
    groups = labeled["group"].to_numpy()
    if len(np.unique(y)) != 2:
        raise ValueError("Both CONFIRMED and FALSE POSITIVE labels are required.")
    if labeled.groupby("label")["group"].nunique().min() < N_SPLITS:
        raise ValueError("Too few distinct host stars in at least one class for 5 folds.")
    print(f"Filtered catalog rows: {len(catalog):,}")
    print(f"Resolved labeled rows: {len(labeled):,}")
    print(f"Unique resolved host stars: {len(np.unique(groups)):,}")
    print("Class counts (0=false positive, 1=confirmed):")
    print(pd.Series(y).value_counts().sort_index())
    print("Features:", list(X.columns))
    return path, raw, catalog, labeled, X, y, groups

def build_model(seed=RANDOM_STATE):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", RandomForestClassifier(
            n_estimators=N_ESTIMATORS,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=seed,
            n_jobs=-1
        ))
    ])

def make_oof(X, y, groups, grouped=True, seed=RANDOM_STATE, n_splits=N_SPLITS):
    splitter = (
        StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        if grouped else
        StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    )
    probs = np.full(len(y), np.nan, dtype=float)
    folds = np.full(len(y), -1, dtype=int)
    split_iter = splitter.split(X, y, groups) if grouped else splitter.split(X, y)
    for fold, (tr, te) in enumerate(split_iter):
        if np.unique(y[tr]).size != 2 or np.unique(y[te]).size != 2:
            raise ValueError(
                f"Fold {fold+1} has a single class. Inspect group distribution; "
                "do not silently report fold-level class metrics."
            )
        if grouped and set(groups[tr]).intersection(set(groups[te])):
            raise RuntimeError(f"Host-star overlap in fold {fold+1}.")
        model = build_model(seed=seed)
        model.fit(X.iloc[tr], y[tr])
        probs[te] = model.predict_proba(X.iloc[te])[:, 1]
        folds[te] = fold
    if np.isnan(probs).any() or (folds < 0).any():
        raise RuntimeError("Every row must receive exactly one out-of-fold prediction.")
    return probs, folds

def metric_dict(y, p, threshold=THRESHOLD):
    pred = (p >= threshold).astype(int)
    return {
        "ROC_AUC": roc_auc_score(y, p),
        "average_precision": average_precision_score(y, p),
        "balanced_accuracy": balanced_accuracy_score(y, pred),
        "Brier_score_lower_is_better": brier_score_loss(y, p),
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0),
        "F1": f1_score(y, pred, zero_division=0),
        "threshold": threshold
    }

def save_csv(df, filename):
    os.makedirs("/kaggle/working", exist_ok=True)
    path = os.path.join("/kaggle/working", filename)
    df.to_csv(path, index=False)
    print("Saved:", path)

## Aim
Measure how much the random-row and host-star-grouped validation results vary across several split seeds. Each repeat creates one out-of-fold prediction per labeled row. Repeats are **sensitivity analyses**, not independent datasets.

In [2]:
path, raw, catalog, labeled, X, y, groups = load_main_sample()

SEEDS = [42, 52, 62, 72, 82]
records = []
oof_frames = []

for seed in SEEDS:
    print(f"\n=== Seed {seed} ===")
    random_p, random_fold = make_oof(X, y, groups, grouped=False, seed=seed)
    grouped_p, grouped_fold = make_oof(X, y, groups, grouped=True, seed=seed)

    for design, probs, fold_ids in [
        ("Random row-level CV", random_p, random_fold),
        ("Host-star grouped CV", grouped_p, grouped_fold)
    ]:
        row = {"seed": seed, "validation": design, **metric_dict(y, probs)}
        records.append(row)
        if design == "Host-star grouped CV":
            # Confirm no host appears in both train and test for every fold.
            for f in range(N_SPLITS):
                test_g = set(groups[fold_ids == f])
                train_g = set(groups[fold_ids != f])
                assert not test_g.intersection(train_g), f"Overlap in seed {seed}, fold {f}"
        oof_frames.append(pd.DataFrame({
            "seed": seed, "validation": design,
            "row_index": labeled.index, "host_id": groups,
            "true_label": y, "oof_probability": probs, "fold": fold_ids
        }))

repeat_metrics = pd.DataFrame(records)
display(repeat_metrics.round(4))
save_csv(repeat_metrics, "10_repeated_validation_metrics.csv")
save_csv(pd.concat(oof_frames, ignore_index=True), "10_repeated_oof_predictions.csv")

Using catalog: /kaggle/input/datasets/sairakeshnagaram/koi-dataset/cumulative_2026.10.20_19.25.07.csv
Filtered catalog rows: 4,249
Resolved labeled rows: 3,195
Unique resolved host stars: 2,654
Class counts (0=false positive, 1=confirmed):
0    1229
1    1966
Name: count, dtype: int64
Features: ['koi_period', 'koi_time0bk', 'koi_impact', 'koi_duration', 'koi_depth', 'koi_prad', 'koi_teq', 'koi_insol', 'koi_steff', 'koi_slogg', 'koi_srad', 'koi_kepmag']

=== Seed 42 ===

=== Seed 52 ===

=== Seed 62 ===

=== Seed 72 ===

=== Seed 82 ===


,seed,validation,ROC_AUC,average_precision,balanced_accuracy,Brier_score_lower_is_better,precision,recall,F1,threshold
0,42,Random row-level CV,0.9262,0.9445,0.8471,0.1036,0.8671,0.9196,0.8926,0.5
1,42,Host-star grouped CV,0.9236,0.9438,0.8450,0.1057,0.8676,0.9130,0.8897,0.5
2,52,Random row-level CV,0.9278,0.9479,0.8456,0.1032,0.8668,0.9166,0.8910,0.5
3,52,Host-star grouped CV,0.9216,0.9414,0.8404,0.1062,0.8626,0.9135,0.8874,0.5
4,62,Random row-level CV,0.9259,0.9456,0.8449,0.1042,0.8656,0.9176,0.8909,0.5
5,62,Host-star grouped CV,0.9237,0.9431,0.8400,0.1057,0.8647,0.9069,0.8853,0.5
6,72,Random row-level CV,0.9260,0.9457,0.8432,0.1040,0.8665,0.9110,0.8882,0.5
7,72,Host-star grouped CV,0.9251,0.9450,0.8449,0.1048,0.8679,0.9120,0.8894,0.5
8,82,Random row-level CV,0.9255,0.9438,0.8451,0.1042,0.8651,0.9196,0.8915,0.5
9,82,Host-star grouped CV,0.9248,0.9444,0.8434,0.1050,0.8668,0.9105,0.8881,0.5


Saved: /kaggle/working/10_repeated_validation_metrics.csv
Saved: /kaggle/working/10_repeated_oof_predictions.csv


In [3]:
# Summarize variation across split seeds. SD describes split sensitivity;
# it is not a confidence interval for performance on all possible stars.
metric_names = [
    "ROC_AUC", "average_precision", "balanced_accuracy",
    "Brier_score_lower_is_better", "precision", "recall", "F1"
]
stability = (
    repeat_metrics.groupby("validation")[metric_names]
    .agg(["mean", "std", "min", "max"])
    .reset_index()
)
display(stability.round(4))
save_csv(stability, "10_validation_stability_summary.csv")

# Paired grouped-minus-random differences for each seed.
wide = repeat_metrics.pivot(index="seed", columns="validation", values=metric_names)
diff_rows = []
for seed in SEEDS:
    row = {"seed": seed}
    for metric in metric_names:
        row[metric + "_grouped_minus_random"] = (
            wide.loc[seed, (metric, "Host-star grouped CV")]
            - wide.loc[seed, (metric, "Random row-level CV")]
        )
    diff_rows.append(row)
differences = pd.DataFrame(diff_rows)
display(differences.round(4))
save_csv(differences, "10_paired_grouped_minus_random.csv")

validation ROC_AUC                         average_precision  \
                           mean     std     min     max              mean   
0  Host-star grouped CV  0.9238  0.0014  0.9216  0.9251            0.9435   
1   Random row-level CV  0.9263  0.0009  0.9255  0.9278            0.9455   

                          balanced_accuracy  ... precision          recall  \
      std     min     max              mean  ...       min     max    mean   
0  0.0014  0.9414  0.9450            0.8427  ...    0.8626  0.8679  0.9112   
1  0.0016  0.9438  0.9479            0.8452  ...    0.8651  0.8671  0.9169   

                               F1                          
      std     min     max    mean     std     min     max  
0  0.0027  0.9069  0.9135  0.8880  0.0018  0.8853  0.8897  
1  0.0036  0.9110  0.9196  0.8908  0.0016  0.8882  0.8926  

[2 rows x 29 columns]

Saved: /kaggle/working/10_validation_stability_summary.csv


,seed,ROC_AUC_grouped_minus_random,average_precision_grouped_minus_random,balanced_accuracy_grouped_minus_random,Brier_score_lower_is_better_grouped_minus_random,precision_grouped_minus_random,recall_grouped_minus_random,F1_grouped_minus_random
0,42,-0.0026,-0.0008,-0.0021,0.0020,0.0004,-0.0066,-0.0029
1,52,-0.0061,-0.0065,-0.0052,0.0031,-0.0041,-0.0031,-0.0036
2,62,-0.0022,-0.0024,-0.0049,0.0015,-0.0009,-0.0107,-0.0056
3,72,-0.0009,-0.0007,0.0017,0.0008,0.0014,0.0010,0.0012
4,82,-0.0007,0.0006,-0.0017,0.0008,0.0018,-0.0092,-0.0034


Saved: /kaggle/working/10_paired_grouped_minus_random.csv


## Interpretation guardrails
- A small random-versus-grouped difference is still a result; do not describe it as a large leakage effect.
- Report the mean, standard deviation, and range across seeds as split sensitivity.
- The repeated runs share the same catalog rows, so they are not independent replications.
- Grouped validation estimates performance on unseen host stars more directly than row-random validation.